In [4]:
# Manejo de datos
import pandas as pd
import numpy as np

# Leer archivos RData
import pyreadr

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Modelado
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [6]:
result = pyreadr.read_r('../data/listings.RData')
df = list(result.values())[0]

df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


In [7]:
df['price'].head()

0     $97.00
1    $160.00
2     $38.00
3    $145.00
4     $58.00
Name: price, dtype: object

In [9]:
df['price'] = df['price'].replace('[\$,]', '', regex=True)

In [10]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')

In [11]:
df['price'].isnull().sum()

np.int64(95502)

In [12]:
df = df.dropna(subset=['price'])

In [13]:
df['price'].describe()

count    76246.000000
mean       750.509220
std       4250.606945
min          8.000000
25%        120.000000
50%        193.000000
75%        326.000000
max      50123.000000
Name: price, dtype: float64

In [14]:
df['precio_cat'] = pd.qcut(df['price'], q=3, labels=['barato', 'medio', 'caro'])

C:\Users\alexa\AppData\Local\Temp\ipykernel_4468\317413179.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['precio_cat'] = pd.qcut(df['price'], q=3, labels=['barato', 'medio', 'caro'])


In [15]:
df['precio_cat'].value_counts()

precio_cat
barato    25689
caro      25404
medio     25153
Name: count, dtype: int64

## 1. Variables dicotómicas

In [16]:
df['es_caro'] = (df['precio_cat'] == 'caro').astype(int)
df['es_medio'] = (df['precio_cat'] == 'medio').astype(int)
df['es_barato'] = (df['precio_cat'] == 'barato').astype(int)

C:\Users\alexa\AppData\Local\Temp\ipykernel_4468\3806297217.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['es_caro'] = (df['precio_cat'] == 'caro').astype(int)
C:\Users\alexa\AppData\Local\Temp\ipykernel_4468\3806297217.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['es_medio'] = (df['precio_cat'] == 'medio').astype(int)
C:\Users\alexa\AppData\Local\Temp\ipykernel_4468\3806297217.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [17]:
df[['precio_cat', 'es_caro', 'es_medio', 'es_barato']].head()

,precio_cat,es_caro,es_medio,es_barato
0,barato,0,0,1
1,medio,0,1,0
2,barato,0,0,1
3,medio,0,1,0
4,barato,0,0,1


## 2.Train/Test

In [25]:
X = df[['accommodates', 'bedrooms', 'bathrooms']]
y = df['es_caro']

In [26]:
X = X.dropna()
y = y[X.index]

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [28]:
X.isnull().sum()

accommodates    0
bedrooms        0
bathrooms       0
dtype: int64

In [21]:
X = X.dropna()
y = y[X.index]

In [22]:
X.isnull().sum()

accommodates    0
bedrooms        0
bathrooms       0
dtype: int64

## 3. Modelo de regresión logística

In [29]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [30]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train, y_train, cv=5)

print("Accuracy por fold:", scores)
print("Accuracy promedio:", scores.mean())

Accuracy por fold: [0.7695845  0.76646412 0.7626047  0.75831486 0.75971093]
Accuracy promedio: 0.7633358191129302
